# Module 03 — Lesson 05
# Random Variables and Distributions

---

> Lesson 04 ended with a question: "how many of these 10 emails are spam?" That's not a single probability anymore — it's a whole RANGE of possible outcomes (0 spam, 1 spam, 2 spam, ..., 10 spam), each with its own probability. A **random variable** is the tool that captures this entire picture in one object.

This idea shows up everywhere: the number of defective items in a batch of 20, the number of customers who click "buy" out of 100 shown an ad, the number of heads in 5 coin flips. All of these are counting something uncertain — and once you can model them, you can answer questions like "what's the probability at least 6 out of 10 pass this exam by guessing?" This is the mathematical backbone behind A/B testing, quality control, and classification model evaluation.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Define a random variable and distinguish discrete from continuous
- Build a Probability Mass Function (PMF) from scratch for a discrete random variable
- Compute the expected value (mean) and variance of a random variable
- Understand the Bernoulli distribution as the building block of a single yes/no trial
- Compute Binomial probabilities from scratch and verify them with simulation
- Explain the difference between a PMF (discrete) and a PDF (continuous), and why P(X = exact value) = 0 for continuous variables

---
## 1. What Is a Random Variable?

A **random variable** assigns a number to every outcome of a random process. By convention, a capital letter (X) denotes the variable itself — the whole random process — while a lowercase letter (x) denotes one specific observed value.

- X = "sum of two dice" → possible values: 2, 3, ..., 12
- Y = "number of heads in 3 coin flips" → possible values: 0, 1, 2, 3

In [ ]:
import itertools

# Random variable X = sum of two dice
dice_outcomes = list(itertools.product(range(1, 7), range(1, 7)))
X_dice = [d1 + d2 for d1, d2 in dice_outcomes]

# Random variable Y = number of heads in 3 coin flips
coin_outcomes = list(itertools.product(["H", "T"], repeat=3))
Y_heads = [outcome.count("H") for outcome in coin_outcomes]

print(f"Dice outcomes (first 5)     : {dice_outcomes[:5]}")
print(f"X = sum, first 5 values     : {X_dice[:5]}")
print()
print(f"Coin outcomes (all 8)       : {coin_outcomes}")
print(f"Y = number of heads         : {Y_heads}")
print()
print("The random variable maps each raw outcome to a NUMBER we can do math with.")

---
## 2. Discrete vs Continuous Random Variables

| Type | Definition | Examples |
|---|---|---|
| **Discrete** | Countable set of possible values | Number of heads, dice sum, number of defective items |
| **Continuous** | Any value in a range (uncountably infinite) | Height, temperature, time until failure |

This lesson focuses on **discrete** random variables. Lesson 06 covers the most important **continuous** distribution — the Normal Distribution — but we'll take a first look at continuous variables near the end of this lesson.

---
## 3. The Probability Mass Function (PMF)

The **PMF** gives P(X = x) — the probability the random variable takes each specific value. A valid PMF must satisfy two rules:

1. Every probability is $\geq 0$
2. All probabilities **sum to exactly 1**

In [ ]:
from collections import Counter

coin_outcomes = list(itertools.product(["H", "T"], repeat=3))
Y_heads = [outcome.count("H") for outcome in coin_outcomes]

counts = Counter(Y_heads)
total = len(Y_heads)

pmf = {y: count / total for y, count in sorted(counts.items())}

print("PMF of Y = number of heads in 3 coin flips:")
print(f"  {'y':>3} {'P(Y=y)':>10}")
print("  " + "-" * 16)
for y, p in pmf.items():
    print(f"  {y:>3} {p:>10.4f}")

print()
print(f"Sum of all probabilities: {sum(pmf.values()):.4f}  <- must equal 1.0")

---
## 4. Expected Value — The Mean of a Random Variable

$$E[X] = \sum_x x \cdot P(X = x)$$

Expected value is the **long-run average** if you repeated the random process infinitely many times — not a prediction of any single outcome.

In [ ]:
def expected_value(pmf):
    '''Compute E[X] given a PMF dict {value: probability}.'''
    return sum(x * p for x, p in pmf.items())

e_y = expected_value(pmf)
print(f"E[Y] = expected number of heads in 3 flips = {e_y:.2f}")
print()
print("Makes sense: with a fair coin, you'd expect 1.5 heads on average across")
print("many repeats of 3 flips — even though 1.5 heads can never actually happen")
print("in a single trial. The expected value is a LONG-RUN AVERAGE, not a")
print("prediction of any single outcome.")

---
## 5. Variance and Standard Deviation of a Random Variable

$$Var(X) = \sum_x (x - E[X])^2 \cdot P(X = x)$$

Same idea as Lesson 02, applied to a random variable instead of a raw dataset.

In [ ]:
import math

def variance_rv(pmf):
    '''Compute Var(X) given a PMF dict {value: probability}.'''
    mu = expected_value(pmf)
    return sum((x - mu) ** 2 * p for x, p in pmf.items())

var_y = variance_rv(pmf)
std_y = math.sqrt(var_y)

print(f"E[Y]       : {e_y:.3f}")
print(f"Var(Y)     : {var_y:.3f}")
print(f"Std dev(Y) : {std_y:.3f}")
print()
print("Same interpretation as Lesson 02 — std dev tells you how much Y typically")
print("deviates from its expected value across repeated trials.")

---
## 6. The Bernoulli Distribution

The simplest random variable: a **single trial** with two outcomes — success (1) or failure (0) — where success happens with probability p.

$$E[X] = p \qquad Var(X) = p(1-p)$$

In [ ]:
def bernoulli_pmf(p):
    '''PMF for a single Bernoulli(p) trial.'''
    return {0: 1 - p, 1: p}

p_click = 0.12   # 12% of users click an ad
pmf_click = bernoulli_pmf(p_click)

e_click = expected_value(pmf_click)
var_click = variance_rv(pmf_click)

print(f"Bernoulli(p={p_click}) — a single ad impression:")
print(f"  PMF: {pmf_click}")
print(f"  E[X]   = {e_click:.3f}  (equals p, as expected)")
print(f"  Var(X) = {var_click:.4f}  (equals p(1-p) = {p_click*(1-p_click):.4f})")

---
## 7. The Binomial Distribution

**n independent Bernoulli(p) trials**, where X = number of successes. This models exactly the "10 emails, each independently spam or not" scenario from Lesson 04.

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

Requirements: a **fixed** number of trials n, trials that are **independent**, and a **constant** success probability p across every trial.

In [ ]:
def binomial_pmf(n, p, k):
    '''P(X = k) for X ~ Binomial(n, p).'''
    return math.comb(n, k) * (p ** k) * ((1 - p) ** (n - k))

def binomial_distribution(n, p):
    '''Full PMF dict for X ~ Binomial(n, p), for k = 0..n.'''
    return {k: binomial_pmf(n, p, k) for k in range(n + 1)}


# Lesson 04's teaser: 10 emails, each independently 20% likely to be spam
n_emails, p_spam = 10, 0.20
pmf_spam = binomial_distribution(n_emails, p_spam)

print(f"X ~ Binomial(n={n_emails}, p={p_spam}) — number of spam emails out of 10")
print(f"  {'k':>3} {'P(X=k)':>10}")
print("  " + "-" * 16)
for k, p in pmf_spam.items():
    bar = "#" * int(p * 100)
    print(f"  {k:>3} {p:>10.4f}  {bar}")

print()
print(f"Sum of all probabilities: {sum(pmf_spam.values()):.4f}")
print(f"E[X] = n*p = {n_emails * p_spam:.1f}   (matches formula: expect 2 spam emails)")

---
## 8. Simulating the Binomial Distribution

In [ ]:
import random
random.seed(3)

def simulate_binomial(n, p, trials=100_000):
    '''Simulate `trials` repeats of n Bernoulli(p) trials, return experimental PMF.'''
    results = []
    for _ in range(trials):
        successes = sum(1 for _ in range(n) if random.random() < p)
        results.append(successes)

    counts = Counter(results)
    return {k: count / trials for k, count in sorted(counts.items())}


simulated_pmf = simulate_binomial(n_emails, p_spam)

print(f"{'k':>3} {'Theoretical':>12} {'Simulated':>12}")
print("-" * 30)
for k in range(n_emails + 1):
    theo = pmf_spam.get(k, 0)
    sim = simulated_pmf.get(k, 0)
    print(f"{k:>3} {theo:>12.4f} {sim:>12.4f}")

print()
print("100,000 simulated trials closely match the theoretical binomial formula —")
print("this confirms the formula from Section 7 is correct.")

---
## 9. A First Look at Continuous Random Variables

For a continuous random variable, there are **infinitely many** possible values in any range, so we can't list a PMF. Instead we use a **Probability Density Function (PDF)** — a curve where **area under the curve** (not the height itself) gives probability.

$$P(a \leq X \leq b) = \text{area under the density curve between } a \text{ and } b$$

In [ ]:
# A continuous random variable: adult height in cm, roughly bell-shaped.
# We can't list every possible height (infinitely many) — but we CAN approximate
# the shape with a fine grid and sum density * bin_width to estimate area.

def normal_like_density(x, mean=170, std=7):
    '''A bell-curve-shaped density function (preview of Lesson 06).'''
    coeff = 1 / (std * math.sqrt(2 * math.pi))
    exponent = -((x - mean) ** 2) / (2 * std ** 2)
    return coeff * math.exp(exponent)

# Approximate P(168 <= height <= 172) by summing density * bin_width over a fine grid
bin_width = 0.01
n_bins = int((172 - 168) / bin_width)
heights = [168 + i * bin_width for i in range(n_bins)]
area = sum(normal_like_density(h) * bin_width for h in heights)

print(f"P(168 <= height <= 172) approx {area:.4f}   <- AREA under the curve")
print()
print("P(height == exactly 170.000...) is essentially 0 — even though 170cm is")
print("the single MOST LIKELY height, in a continuous distribution no exact")
print("value has positive probability. A single point has zero width, so its")
print("area (probability) is zero.")
print()
print("This is why, for continuous variables, we always ask about a RANGE")
print("('what's P(a <= X <= b)?'), never a single exact value.")
print("Lesson 06 explores this bell curve — the Normal Distribution — in depth.")

---
## 10. Built-in Verification

In [ ]:
# Verify our from-scratch expected_value() and variance_rv() functions
# against the known closed-form Binomial formulas: E[X]=np, Var(X)=np(1-p)

e_ours = expected_value(pmf_spam)
var_ours = variance_rv(pmf_spam)

e_formula = n_emails * p_spam
var_formula = n_emails * p_spam * (1 - p_spam)

print(f"{'Measure':<10} {'From PMF':>12} {'Closed-form':>12}  Match?")
print("-" * 45)
match_e = abs(e_ours - e_formula) < 1e-9
match_v = abs(var_ours - var_formula) < 1e-9
print(f"{'E[X]':<10} {e_ours:>12.4f} {e_formula:>12.4f}   {'match' if match_e else 'MISMATCH'}")
print(f"{'Var(X)':<10} {var_ours:>12.4f} {var_formula:>12.4f}   {'match' if match_v else 'MISMATCH'}")

---
## ⚠️ Common Mistakes

In [ ]:
# -- Mistake 1: Confusing the random variable with an observed value ---------

print("Mistake 1 — X (the variable) vs x (an observed value):")
print("  WRONG notation: 'P(X)'      -- meaningless, X isn't a single number")
print("  RIGHT notation: 'P(X = 3)'  -- the probability the random variable")
print("                   takes the specific value 3")
print()
print("  X describes the ENTIRE random process (all possible outcomes and their")
print("  probabilities). x is just ONE observed or possible value. Always ask")
print("  'the probability that X equals WHAT?' before writing P(X = ...).")

In [ ]:
# -- Mistake 2: Forgetting to verify a PMF sums to 1 --------------------------

# A bug: forgot to include one outcome when building this PMF
buggy_pmf = {0: 0.5, 1: 0.3}   # should have included P(X=2) too!

total = sum(buggy_pmf.values())
print("Mistake 2 — An invalid PMF (doesn't sum to 1):")
print(f"  {buggy_pmf}")
print(f"  Sum = {total:.2f}  <- should be 1.0, something is MISSING or wrong")
print()
print("  Always sanity-check: sum(pmf.values()) should equal 1.0 (within")
print("  floating-point rounding). If it doesn't, you've miscounted outcomes")
print("  or left one out.")

In [ ]:
# -- Mistake 3: Using Binomial when trials aren't independent -----------------

# WRONG: modeling "2 kings drawn from a deck WITHOUT replacement" as Binomial
n, p = 2, 4 / 52
wrong_binomial_p = binomial_pmf(n, p, 2)   # assumes independent draws, WRONG

# CORRECT: account for the changed deck after the first draw (like Lesson 04)
correct_p = (4 / 52) * (3 / 51)

print("Mistake 3 — Binomial requires INDEPENDENT trials with CONSTANT p:")
print(f"  Binomial formula (wrongly assumes replacement) : {wrong_binomial_p:.4f}")
print(f"  Correct (accounts for no replacement)           : {correct_p:.4f}")
print()
print("  Sampling WITHOUT replacement changes p after each draw — that's a")
print("  different distribution (Hypergeometric), not Binomial. Always check:")
print("  are the trials truly independent, and does p stay constant?")

In [ ]:
# -- Mistake 4: Treating a PDF value as if it were a probability --------------

density_at_peak = normal_like_density(170, mean=170, std=2)   # narrow std dev

print("Mistake 4 — A density value is NOT a probability:")
print(f"  Density at the peak (std=2): {density_at_peak:.4f}")
print()
print("  Notice this can exceed 0.1, and with an even narrower std dev it could")
print("  exceed 1 — something that would be illegal for a probability. Densities")
print("  can be ANY non-negative number; only the AREA under the curve (over a")
print("  range) is a probability, and that area is always between 0 and 1.")

---
## ✏️ Practice Exercises

### Exercise 1 — Starter: PMF for the Sum of Two Dice

Build the PMF for X = sum of two dice (values 2 through 12). Print it as a table and verify the probabilities sum to 1.

```python
import itertools
dice_outcomes = list(itertools.product(range(1, 7), range(1, 7)))
X_dice = [d1 + d2 for d1, d2 in dice_outcomes]
```

In [ ]:
import itertools
dice_outcomes = list(itertools.product(range(1, 7), range(1, 7)))
X_dice = [d1 + d2 for d1, d2 in dice_outcomes]
# Your code here

### Exercise 2 — Expected Value and Variance of the Dice Sum

Using the PMF you built in Exercise 1, compute E[X] and Var(X) for the sum of two dice using `expected_value()` and `variance_rv()`. Does E[X] match your intuition (hint: it should be the midpoint of the range 2-12)?

In [ ]:
# Your code here

### Exercise 3 — Quality Control: Binomial Probability Calculator

A factory's items have a 5% defect rate. In a random batch of 20 items:
1. Compute P(exactly 2 defective) using `binomial_pmf()`
2. Compute P(at most 2 defective) by summing `binomial_pmf(20, 0.05, k)` for k = 0, 1, 2

In [ ]:
n, p = 20, 0.05
# Your code here

### Exercise 4 — Simulate the Factory Scenario

Using `simulate_binomial()`, simulate the Exercise 3 scenario (n=20, p=0.05) with 50,000 trials. Compare the simulated P(exactly 2 defective) and P(at most 2 defective) to your theoretical answers from Exercise 3.

In [ ]:
# Your code here

### Exercise 5 — (Challenge) Guessing on a Multiple-Choice Test

A test has 10 questions, each with 4 options (so random guessing gives a 25% chance of a correct answer per question). Model the number of correct guesses as X ~ Binomial(n=10, p=0.25).

1. Build the full PMF using `binomial_distribution()`
2. Compute P(X >= 6) — the probability of "passing" by guessing alone (getting 6 or more correct)
3. Print a one-sentence conclusion about whether guessing is a viable strategy

In [ ]:
n, p = 10, 0.25
# Your code here

---
## 📌 Key Takeaways

- **A random variable maps outcomes to numbers, and its PMF describes the full probability picture at once.** Every valid PMF must have probabilities that are non-negative and sum to exactly 1 — that check catches most bugs when you build one from scratch.

- **Expected value and variance generalize the mean and variance from Lesson 02 to random variables.** E[X] is the long-run average across infinitely many repeats (not a prediction for any single trial), and Var(X)/its square root (std dev) measure how much outcomes typically deviate from that average.

- **The Binomial distribution models "how many successes in n independent yes/no trials," and it's everywhere in data science** — quality control (defect rates), A/B testing (conversion counts), and evaluating classifiers (how many predictions are correct out of n). It requires independent trials with a constant success probability; when that assumption breaks (like sampling without replacement), you need a different model.

---
## What's Next?

**Lesson 06 — The Normal Distribution**
You've now built discrete distributions from scratch and taken a first look at continuous ones with the density function `normal_like_density()`. Next, you'll dive deep into the single most important continuous distribution in all of statistics — the Normal (bell curve) Distribution — and learn why it shows up almost everywhere, tying directly back to the z-scores from Lesson 02.